# Taller 2. PP-StructureV3: layout, orden de lectura y OCR

**Qué produce:** las visualizaciones del propio pipeline (regiones, orden de lectura, líneas de texto), la transcripción en Markdown, el JSON completo y una tabla de bloques en el orden en que el pipeline los leyó.

**Para qué:** observar qué ocurre con el **orden de lectura** en una página de varias columnas y separar dos tipos de error: los de **segmentación** (bloques mal delimitados o mal ordenados) y los de **reconocimiento** (caracteres mal leídos).

**Entorno:** Colab con GPU T4 recomendada. Sin GPU funciona, pero tarda varios minutos por página. La primera ejecución descarga los modelos.

## 0. Instalar dependencias (solo Colab)

Se instala la versión de PaddlePaddle que corresponde al entorno: con GPU, `paddlepaddle-gpu` desde el repositorio oficial de Paddle; sin GPU, la versión para CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    import shutil

    GPU = shutil.which("nvidia-smi") is not None
    if GPU:
        paddle = ["paddlepaddle-gpu==3.2.1", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"]
    else:
        paddle = ["paddlepaddle==3.2.1"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paddle], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paddleocr[doc-parser]==3.7.0"], check=True)
    # PaddlePaddle-GPU instala la biblioteca NCCL de CUDA 12 sobre la de CUDA 13 que usa el PyTorch de Colab,
    # y PaddleOCR importa PyTorch: se restituye la versión que pide PyTorch (compatible con Paddle).
    import importlib.metadata as metadata

    nccl = [r.split(";")[0].replace(" ", "") for r in (metadata.requires("torch") or []) if r.startswith("nvidia-nccl")]
    if nccl:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", *nccl], check=True)
    print("Instalado PaddlePaddle", "GPU" if GPU else "CPU", "y PaddleOCR 3.7.0")
else:
    print("Fuera de Colab no se instala nada: preparar el entorno según cuadernos/README.md.")

## 1. Elegir las imágenes

Por defecto se procesa la **página común del taller** (*Centro Gallego*, Montevideo, 1917): la misma página en los cuatro cuadernos permite comparar modelos. Cambiá `SELECCION` para trabajar con otras fuentes:

| Valor | Qué procesa |
|---|---|
| `"comun"` | la página común (recomendado para la primera pasada) |
| `"prensa"` | todas las páginas de prensa del taller |
| `"manuscritos"` | cartas, actas y otros manuscritos del taller |
| `"todas"` | prensa y manuscritos |
| `"propias"` | tus imágenes; en Colab se abre un diálogo para subirlas, fuera de Colab se leen de `mis_imagenes/` |

**No subas material sensible o con restricciones de uso** a servicios externos.

In [ ]:
SELECCION = "comun"

In [ ]:
import csv
import sys
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display
from PIL import Image

EN_COLAB = "google.colab" in sys.modules
REPO_RAW = "https://raw.githubusercontent.com/agusnieto77/sicss-uruguay-2026/main/cuadernos/imagenes/"
BASE = Path("/content/taller") if EN_COLAB else Path.cwd()
CARPETA_IMAGENES = BASE / "imagenes"
EXTENSIONES = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}


def traer(relativa):
    """Ruta local de un archivo del taller; en Colab lo descarga del repositorio la primera vez."""
    destino = CARPETA_IMAGENES / relativa
    if EN_COLAB and not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + relativa, destino)
    if not destino.exists():
        raise FileNotFoundError(f"No encuentro {destino}. Abrí el cuaderno desde la carpeta cuadernos/ del repositorio.")
    return destino


def mostrar(imagen, ancho=1000):
    """Muestra una imagen PIL reducida a un ancho cómodo para la pantalla."""
    if imagen.width > ancho:
        imagen = imagen.resize((ancho, round(imagen.height * ancho / imagen.width)))
    display(imagen)


FICHAS = {}
if SELECCION == "propias":
    carpeta_propia = BASE / "mis_imagenes"
    carpeta_propia.mkdir(parents=True, exist_ok=True)
    if EN_COLAB:
        from google.colab import files

        for nombre, datos in files.upload().items():
            if Path(nombre).suffix.lower() in EXTENSIONES:
                (carpeta_propia / Path(nombre).name).write_bytes(datos)
            else:
                print("Ignorado (no es imagen):", nombre)
    IMAGENES = sorted(p for p in carpeta_propia.iterdir() if p.suffix.lower() in EXTENSIONES)
else:
    with open(traer("fuentes.csv"), encoding="utf-8-sig") as archivo:
        fichas = list(csv.DictReader(archivo, delimiter=";"))
    filtros = {
        "comun": lambda f: f["pagina_comun"] == "si",
        "prensa": lambda f: f["tipo"] == "prensa",
        "manuscritos": lambda f: f["tipo"] != "prensa",
        "todas": lambda f: True,
    }
    if SELECCION not in filtros:
        raise ValueError(f"SELECCION debe ser uno de {sorted(filtros) + ['propias']}")
    elegidas = [f for f in fichas if filtros[SELECCION](f)]
    IMAGENES = [traer(f["archivo"]) for f in elegidas]
    FICHAS = {ruta.name: ficha for ruta, ficha in zip(IMAGENES, elegidas)}
    display(pd.DataFrame(elegidas)[["archivo", "tipo", "titulo", "fecha", "lugar", "institucion", "licencia"]])

assert IMAGENES, "No hay imágenes para procesar: revisá SELECCION o la carpeta mis_imagenes/."
print(f"{len(IMAGENES)} imagen(es):", [p.name for p in IMAGENES])

## 2. Preparar el pipeline

`IDIOMA = "es"` usa el reconocedor de alfabeto latino de PP-OCRv5, que contempla tildes y eñes. Sin ese parámetro el pipeline usa un reconocedor entrenado sobre todo con chino e inglés. Se desactivan los módulos que no se usan en el taller (tablas, fórmulas, sellos, gráficos).

In [ ]:
import time


def con_reintentos(funcion, intentos=3, espera=10):
    """Repite una operación que descarga archivos de servidores remotos (modelos, tipografías)."""
    for intento in range(1, intentos + 1):
        try:
            return funcion()
        except Exception as error:  # cortes de red durante descargas
            if intento == intentos:
                raise
            print(f"Intento {intento} falló ({type(error).__name__}); reintento en {espera} s")
            time.sleep(espera)

In [ ]:
import paddle
from paddleocr import PPStructureV3

DISPOSITIVO = "gpu:0" if paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0 else "cpu"
PADDLE_INSTALADO = "paddlepaddle-gpu" if paddle.device.is_compiled_with_cuda() else "paddlepaddle"
IDIOMA = "es"
SALIDA = BASE / "resultados" / "02_pp_structurev3"
SALIDA.mkdir(parents=True, exist_ok=True)

pipeline = con_reintentos(lambda: PPStructureV3(
    lang=IDIOMA,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_table_recognition=False,
    use_formula_recognition=False,
    use_seal_recognition=False,
    use_chart_recognition=False,
    device=DISPOSITIVO,
))
print("Dispositivo:", DISPOSITIVO)

## 3. Procesar las páginas

In [ ]:
def bloques_en_orden(resultado, pagina):
    """Tabla con los bloques en el orden de lectura que reconstruyó el pipeline."""
    datos = resultado.json.get("res", resultado.json)
    filas = []
    for posicion, bloque in enumerate(datos.get("parsing_res_list", []), 1):
        x1, y1, x2, y2 = (round(float(v)) for v in bloque.get("block_bbox", [0, 0, 0, 0]))
        filas.append({
            "pagina": pagina.name,
            "orden": posicion,
            "clase": bloque.get("block_label"),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "texto": bloque.get("block_content", ""),
            "evaluacion_humana": "",
        })
    return filas

In [ ]:
VISTAS = {
    "layout_det_res": "Regiones detectadas",
    "layout_order_res": "Orden de lectura reconstruido",
    "overall_ocr_res": "Líneas de texto reconocidas",
}
bloques = []
for pagina in IMAGENES:
    destino = SALIDA / pagina.stem
    destino.mkdir(parents=True, exist_ok=True)
    for resultado in pipeline.predict(str(pagina)):
        imagenes = con_reintentos(lambda: resultado.img)  # la primera vez descarga tipografías
        resultado.save_to_img(save_path=str(destino))
        resultado.save_to_json(save_path=str(destino))
        resultado.save_to_markdown(save_path=str(destino))
        print("Página:", pagina.name)
        for clave, titulo in VISTAS.items():
            if clave in imagenes:
                print(titulo)
                mostrar(imagenes[clave])
        bloques += bloques_en_orden(resultado, pagina)
        texto = resultado.markdown.get("markdown_texts", "")
        print("TRANSCRIPCIÓN (primeros 4000 caracteres)\n", str(texto)[:4000])

bloques = pd.DataFrame(bloques, columns=["pagina", "orden", "clase", "x1", "y1", "x2", "y2", "texto", "evaluacion_humana"])
bloques.to_csv(SALIDA / "bloques.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
print("Guardado en:", SALIDA)

## 4. Bloques en orden de lectura

Cada fila es un bloque, en el orden en que el pipeline lo incorporó al texto. Compará esta secuencia con la lectura humana de la página.

In [ ]:
pd.set_option("display.max_colwidth", 120)
display(bloques[["pagina", "orden", "clase", "texto"]])

## 5. Discusión

Completá `evaluacion_humana` en `bloques.csv` (se abre directamente en Excel) con `correcto`, `orden_erroneo`, `fusiona`, `fragmenta` o `error_reconocimiento`.

1. ¿Dónde salta el texto de una columna a otra de forma incorrecta? ¿Se mezclan titulares con el cuerpo de la noticia?
2. ¿Los avisos quedan incorporados a un artículo?
3. ¿Qué pasa con nombres propios, fechas y cifras (por ejemplo, la lista de precios)? ¿Se conservan las tildes y la ñ?
4. Separá los errores de **segmentación** de los de **reconocimiento**: ¿cuáles alteran más la fuente?

## Registro de mediaciones

Toda transcripción automática depende de la herramienta, la versión, la configuración y el equipo que la produjeron. Esta celda guarda esos datos en `procesamiento.json`, junto a los resultados, para poder citarlos y reproducirlos.

In [ ]:
import datetime
import importlib.metadata as metadata
import json
import platform

registro = {
    "herramienta": "PP-StructureV3 (PaddleOCR)",
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "entorno": "Google Colab" if EN_COLAB else platform.platform(),
    "python": platform.python_version(),
    "dispositivo": DISPOSITIVO,
    "imagenes": [p.name for p in IMAGENES],
    "versiones": {p: metadata.version(p) for p in ("paddleocr", "paddlex", PADDLE_INSTALADO)},
    "parametros": {"lang": IDIOMA, "tablas": False, "formulas": False, "sellos": False, "graficos": False},
}
(SALIDA / "procesamiento.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(registro, ensure_ascii=False, indent=2))

## Descargar resultados

In [ ]:
import shutil

if EN_COLAB:
    from google.colab import files

    files.download(shutil.make_archive(f"/content/resultados_02_pp_structurev3", "zip", SALIDA))
else:
    print("Resultados en:", SALIDA.resolve())